# Task 2 — Biggest Gains & Declines in Happiness (Slopegraphs, Altair)

Builds the two Task 2 slopegraphs for the **Drivers of Happiness** site:

1. **Biggest gains** — the 5 countries whose life evaluation rose the most over ~20 years
2. **Biggest declines** — the 5 countries whose life evaluation fell the most

**Method note:** single WHR years are noisy, so instead of comparing 2006 to 2025 directly we compare **3-year endpoint windows** (average of 2006–08 vs. average of 2023–25) and require at least 2 observations in each window.

Each exported page gets its own **country picker** — pill buttons for the top 10 movers, with the top 5 selected by default. Endpoint labels are re-dodged in JavaScript whenever the selection changes, so near-tied values never overlap. The charts use the site theme from Task 1, with the green/red poles of the map's color scale encoding direction of change (the pair passes colorblind-separation and contrast checks, and country names are written directly on every line, so color is never the only cue).

**Running in Google Colab:** upload this notebook, run all cells — the data-loading cell will prompt you to upload `whr_2006_2025_cleaned.csv` (it lives in `charts/data/` in the repo). Running locally from the repo root works too and skips the upload. The picker only exists in the exported HTML pages, not in the inline previews.

In [ ]:
%pip install -q "altair>=5.0,<6"

## Load the data

Reads from the repo if the notebook is run locally; otherwise prompts for an upload in Colab.

In [ ]:
import os
import pandas as pd

CSV_PATH = "charts/data/whr_2006_2025_cleaned.csv"  # path when run from the repo root

if os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
elif os.path.exists("whr_2006_2025_cleaned.csv"):
    df_raw = pd.read_csv("whr_2006_2025_cleaned.csv")
else:
    from google.colab import files  # only exists in Colab
    print("Upload whr_2006_2025_cleaned.csv")
    uploaded = files.upload()
    df_raw = pd.read_csv(next(iter(uploaded)))

print(df_raw.shape)
df_raw[["year", "country", "region", "life_evaluation"]].head()

## Find the biggest movers

For every country with enough data in both endpoint windows, compute the change in average life evaluation. The top/bottom **10** feed the pickers; the top/bottom **5** are shown by default. The shared y-domain is computed from the full pool so both panels use the same scale.

In [ ]:
import math

START_LABEL, END_LABEL = "2006–08", "2023–25"
POOL, DEFAULT_N = 10, 5

d = df_raw.dropna(subset=["life_evaluation"])
start = d[d.year.between(2006, 2008)].groupby("country").life_evaluation.agg(["mean", "count"])
end = d[d.year.between(2023, 2025)].groupby("country").life_evaluation.agg(["mean", "count"])

m = start.join(end, lsuffix="_start", rsuffix="_end", how="inner")
m = m[(m.count_start >= 2) & (m.count_end >= 2)]  # ≥2 observations per window
m["change"] = m.mean_end - m.mean_start
m = m.join(d.groupby("country").region.first())

gains = m.nlargest(POOL, "change")
declines = m.nsmallest(POOL, "change")

both = pd.concat([gains, declines])[["mean_start", "mean_end"]]
Y_DOMAIN = [
    math.floor((both.min().min() - 0.15) * 2) / 2,
    math.ceil((both.max().max() + 0.15) * 2) / 2,
]
print("y domain:", Y_DOMAIN)
pd.concat([gains.head(), declines.head()])[["mean_start", "mean_end", "change"]].round(2)

## Shape the data and the initial labels

Each country becomes two rows (one per endpoint). Endpoint labels get a small greedy **dodge** so near-tied values don't overlap. The same dodge is re-implemented in the exported page's JavaScript so labels stay readable as the picker changes.

In [ ]:
def dodge(values, gap=0.17, lo=None, hi=None):
    """Nudge label y-positions apart so near-tied endpoints stay readable."""
    lo, hi = Y_DOMAIN[0] if lo is None else lo, Y_DOMAIN[1] if hi is None else hi
    order = sorted(range(len(values)), key=lambda i: -values[i])
    out = dict.fromkeys(order)
    prev = hi + gap
    for i in order:
        y = min(values[i], prev - gap)
        out[i] = y
        prev = y
    low = min(out.values())
    if low < lo:
        out = {i: y + (lo - low) for i, y in out.items()}
    return [out[i] for i in range(len(values))]


def initial_label_rows(m, default_countries):
    sub = m.loc[default_countries]
    starts = dodge(sub.mean_start.tolist())
    ends = dodge(sub.mean_end.tolist())
    rows = []
    for (country, r), sy, ey in zip(sub.iterrows(), starts, ends):
        rows.append({"period": START_LABEL, "label_y": sy, "text": f"{r.mean_start:.1f}"})
        rows.append({"period": END_LABEL, "label_y": ey, "text": f"{country}  {r.mean_end:.1f}"})
    return rows


def slope_data(m):
    """Long-form rows (two per country), ordered best-to-worst mover."""
    rows = []
    for country, r in m.iterrows():
        common = dict(country=country, region=r.region, change=r.change,
                      start_score=r.mean_start, end_score=r.mean_end)
        rows.append(dict(common, period=START_LABEL, score=r.mean_start))
        rows.append(dict(common, period=END_LABEL, score=r.mean_end))
    return pd.DataFrame(rows)


## Build the slopegraphs

Design choices:
- **One hue per panel** — green for gains, red for declines, the poles of the Task 1 map scale. Identity is carried by direct labels on every line, never by color alone.
- **Shared y-domain** across the two panels, so a steep line in one panel means the same thing in the other.
- **2px lines, white-ringed endpoint dots**, recessive grid, and the site's font/ink via the same `site_theme()` as Task 1.
- The `sel` param holds a `|`-joined list of visible countries; the exported page's picker drives it. Label layers read a **named dataset** (`labels`) that the page rewrites on every change.
- **Tooltips** carry country, region, both window scores, and the signed change.

In [ ]:
import altair as alt

SITE_FONT = '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif'
INK, INK_SOFT, LINE = "#2f3542", "#6b7280", "#ece4d4"
GREEN, RED = "#1a9850", "#d73027"

def site_theme(chart):
    return (
        chart.configure(font=SITE_FONT, background="white")
        .configure_view(stroke=None)
        .configure_title(
            anchor="start", color=INK, fontSize=17, fontWeight=700,
            subtitleColor=INK_SOFT, subtitleFontSize=12, subtitlePadding=6, offset=14,
        )
        .configure_axis(
            labelColor=INK_SOFT, titleColor=INK, labelFontSize=11, titleFontSize=12,
            gridColor=LINE, domainColor=LINE, tickColor=LINE,
        )
    )

def slopegraph(data, color, title, subtitle, default_countries, init_labels):
    sel = alt.param(name="sel", value="|".join(default_countries))
    keep = 'indexof(split(sel, "|"), datum.country) >= 0'

    x = alt.X(
        "period:N", title=None, sort=[START_LABEL, END_LABEL],
        scale=alt.Scale(domain=[START_LABEL, END_LABEL], padding=0.35),
        axis=alt.Axis(labelAngle=0, labelFontSize=12, labelColor=INK, domain=False, ticks=False),
    )
    y = alt.Y("score:Q", title="Life evaluation (0–10)",
              scale=alt.Scale(domain=Y_DOMAIN), axis=alt.Axis(tickCount=5))
    tooltip = [
        alt.Tooltip("country:N", title="Country"),
        alt.Tooltip("region:N", title="Region"),
        alt.Tooltip("start_score:Q", title=START_LABEL, format=".2f"),
        alt.Tooltip("end_score:Q", title=END_LABEL, format=".2f"),
        alt.Tooltip("change:Q", title="Change", format="+.2f"),
    ]

    base = alt.Chart(data)
    lines = base.mark_line(strokeWidth=2, color=color).encode(
        x=x, y=y, detail="country:N", tooltip=tooltip
    ).transform_filter(keep).add_params(sel)
    points = base.mark_point(filled=True, size=70, color=color, stroke="white", strokeWidth=1.5).encode(
        x=x, y=y, detail="country:N", tooltip=tooltip
    ).transform_filter(keep)

    # Label layers read the JS-managed dataset "labels": rows of
    # {period, label_y, text} recomputed (filtered + dodged) on every
    # selection change by the page script.
    labels = alt.Chart(alt.Data(values=init_labels, name="labels"))
    left_vals = labels.transform_filter(f'datum.period == "{START_LABEL}"').mark_text(
        align="right", dx=-10, fontSize=11, color=INK_SOFT
    ).encode(x=x, y=alt.Y("label_y:Q", scale=alt.Scale(domain=Y_DOMAIN)), text="text:N")
    right_names = labels.transform_filter(f'datum.period == "{END_LABEL}"').mark_text(
        align="left", dx=10, fontSize=11, fontWeight=600, color=INK
    ).encode(x=x, y=alt.Y("label_y:Q", scale=alt.Scale(domain=Y_DOMAIN)), text="text:N")

    return (lines + points + left_vals + right_names).properties(
        width=250, height=360,
        padding={"left": 5, "right": 95, "top": 5, "bottom": 5},
        title=alt.Title(title, subtitle=subtitle),
    )

SUBTITLE = "Average life evaluation, 3-year endpoint windows"

gains_default = list(gains.index[:DEFAULT_N])
declines_default = list(declines.index[:DEFAULT_N])

gains_chart = site_theme(slopegraph(
    slope_data(gains), GREEN, "Biggest gains in happiness", SUBTITLE,
    gains_default, initial_label_rows(gains, gains_default)))
declines_chart = site_theme(slopegraph(
    slope_data(declines), RED, "Biggest declines in happiness", SUBTITLE,
    declines_default, initial_label_rows(declines, declines_default)))

gains_chart

In [ ]:
declines_chart

## Export standalone HTML with the country picker

Each chart is wrapped in a small page template that adds the themed pill-button picker (top 10 movers, top 5 on by default, at least one always kept on). Clicking a pill updates the `sel` signal and rewrites the `labels` dataset with freshly dodged positions.

In [ ]:
import json

PAGE_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>__TITLE__</title>
<script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
<style>
  body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
      "Helvetica Neue", Arial, sans-serif;
    margin: 0;
    padding: 14px 16px;
    background: #ffffff;
  }
  #picker { display: flex; flex-wrap: wrap; align-items: center; gap: 6px; margin-bottom: 12px; }
  #picker .picker-label { color: #6b7280; font-size: 12px; margin-right: 2px; }
  #picker button {
    border: 1px solid #ece4d4;
    background: #fffbef;
    color: #6b7280;
    font: inherit;
    font-size: 12px;
    padding: 4px 12px;
    border-radius: 999px;
    cursor: pointer;
    transition: background 0.15s ease, color 0.15s ease, border-color 0.15s ease;
  }
  #picker button:hover { border-color: #f6b93b; }
  #picker button.on {
    background: #f6b93b;
    border-color: #f6b93b;
    color: #ffffff;
    font-weight: 600;
  }
</style>
</head>
<body>
<div id="picker"><span class="picker-label">Countries:</span></div>
<div id="vis"></div>
<script>
  const spec = __SPEC__;
  const COUNTRIES = __COUNTRIES__;   // ordered biggest mover first
  const DEFAULT_N = __DEFAULT_N__;
  const START_LABEL = "__START__", END_LABEL = "__END__";
  const Y_LO = __YLO__, Y_HI = __YHI__, GAP = 0.17;

  const selected = new Set(COUNTRIES.slice(0, DEFAULT_N).map(c => c.country));

  // Same greedy dodge as the notebook: nudge labels apart top-down.
  function dodge(rows) {
    const order = rows.map((r, i) => i).sort((a, b) => rows[b].y - rows[a].y);
    let prev = Y_HI + GAP;
    const out = {};
    for (const i of order) { out[i] = Math.min(rows[i].y, prev - GAP); prev = out[i]; }
    const low = Math.min(...Object.values(out), Y_HI);
    const shift = low < Y_LO ? Y_LO - low : 0;
    return rows.map((r, i) => out[i] + shift);
  }

  function labelRows() {
    const rows = [];
    const active = COUNTRIES.filter(c => selected.has(c.country));
    const starts = dodge(active.map(c => ({ y: c.start })));
    const ends = dodge(active.map(c => ({ y: c.end })));
    active.forEach((c, i) => {
      rows.push({ period: START_LABEL, label_y: starts[i], text: c.start.toFixed(1) });
      rows.push({ period: END_LABEL, label_y: ends[i], text: c.country + "  " + c.end.toFixed(1) });
    });
    return rows;
  }

  vegaEmbed("#vis", spec, { actions: false, renderer: "svg" }).then(({ view }) => {
    const picker = document.getElementById("picker");

    function refresh() {
      view.signal("sel", [...selected].join("|"));
      view.data("labels", labelRows());
      view.runAsync();
    }

    for (const c of COUNTRIES) {
      const btn = document.createElement("button");
      btn.type = "button";
      btn.textContent = c.country;
      if (selected.has(c.country)) btn.classList.add("on");
      btn.addEventListener("click", () => {
        if (selected.has(c.country)) {
          if (selected.size === 1) return;   // keep at least one line
          selected.delete(c.country);
          btn.classList.remove("on");
        } else {
          selected.add(c.country);
          btn.classList.add("on");
        }
        refresh();
      });
      picker.appendChild(btn);
    }

    refresh();
  });
</script>
</body>
</html>
"""


def export(chart, m, filename, title):
    countries = [
        {"country": c, "start": round(r.mean_start, 2), "end": round(r.mean_end, 2)}
        for c, r in m.iterrows()
    ]
    html = (
        PAGE_TEMPLATE
        .replace("__TITLE__", title)
        .replace("__SPEC__", chart.to_json(indent=None))
        .replace("__COUNTRIES__", json.dumps(countries))
        .replace("__DEFAULT_N__", str(DEFAULT_N))
        .replace("__START__", START_LABEL)
        .replace("__END__", END_LABEL)
        .replace("__YLO__", str(Y_DOMAIN[0]))
        .replace("__YHI__", str(Y_DOMAIN[1]))
    )
    with open(filename, "w") as f:
        f.write(html)

export(gains_chart, gains, "task2_slopegraph_gains.html", "Biggest gains in happiness")
export(declines_chart, declines, "task2_slopegraph_declines.html", "Biggest declines in happiness")
print("Saved task2_slopegraph_gains.html and task2_slopegraph_declines.html")

try:
    from google.colab import files
    files.download("task2_slopegraph_gains.html")
    files.download("task2_slopegraph_declines.html")
except ImportError:
    pass  # running locally — files are next to the notebook

## Embedding into the site

Move the two exported files into `charts/`, then embed them side by side in Task 2's `viz-row` in `index.html`:

```html
<div class="viz-row">
  <iframe src="charts/task2_slopegraph_gains.html" class="viz-frame"
          loading="lazy" style="min-height: 580px"
          title="Countries with the biggest gains in happiness"></iframe>
  <iframe src="charts/task2_slopegraph_declines.html" class="viz-frame"
          loading="lazy" style="min-height: 580px"
          title="Countries with the biggest declines in happiness"></iframe>
</div>
```

Still to come for Task 2 (left as placeholders on the site): the full country selector across all countries and the per-country explanatory-factors table.